# Patient exploration

Initial exploration of the PhysioNet 2019 sepsis data.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

## Load one patient

In [ ]:
base_url = "https://physionet.org/files/challenge-2019/1.0.0/training/training_setA/"

patient_id = "p000034"
df = pd.read_csv(base_url + patient_id + ".psv", sep="|")

df

In [ ]:
df[[
    "ICULOS",
    "HR",
    "O2Sat",
    "Temp",
    "SBP",
    "MAP",
    "Resp",
    "SepsisLabel"
]]

## Missing data

In [ ]:
df.isna().sum()

In [ ]:
missing_percentage = df.isna().mean() * 100
missing_percentage.sort_values(ascending=False)

## Heart rate over time

In [ ]:
plt.plot(df["ICULOS"], df["HR"], marker="o")
plt.xlabel("Hours in ICU")
plt.ylabel("Heart Rate (bpm)")
plt.title("Heart Rate over ICU stay")
plt.show()

## Sepsis warning label

In [ ]:
sepsis_start = df.loc[
    df["SepsisLabel"] == 1,
    "ICULOS"
].min()

print(sepsis_start)

In [ ]:
plt.plot(df["ICULOS"], df["HR"], marker="o")

plt.axvline(
    x=sepsis_start,
    linestyle="--",
    label="Sepsis warning label"
)

plt.xlabel("Hours in ICU")
plt.ylabel("Heart Rate (bpm)")
plt.title("Heart Rate and sepsis warning label")
plt.legend()
plt.show()

## Patient-level summary

In [ ]:
patient_id = "p000001"
url = base_url + patient_id + ".psv"
patient = pd.read_csv(url, sep="|")

patient.head()

In [ ]:
patient["SepsisLabel"].max()

In [ ]:
len(patient)  # number of rows for this patient

In [ ]:
mean_hr = patient["HR"].mean()
min_o2 = patient["O2Sat"].min()

print("Mean HR:", mean_hr)
print("Minimum O2 saturation:", min_o2)

In [ ]:
summary = {
    "patient": patient_id,
    "hours_in_icu": patient["ICULOS"].max(),
    "septic": patient["SepsisLabel"].max(),
    "mean_HR": patient["HR"].mean(),
    "min_O2Sat": patient["O2Sat"].min(),
    "mean_Resp": patient["Resp"].mean(),
    "mean_MAP": patient["MAP"].mean()
}

summary

## First 20 patients

Quick comparison only. `septic = 0` means no positive `SepsisLabel` in the file, not that the patient is healthy.

In [ ]:
patients_summary = []

for i in range(1, 21):
    patient_id = f"p{i:06d}"
    url = base_url + patient_id + ".psv"
    patient = pd.read_csv(url, sep="|")

    summary = {
        "patient": patient_id,
        "hours_in_icu": patient["ICULOS"].max(),
        "septic": patient["SepsisLabel"].max(),
        "mean_HR": patient["HR"].mean(),
        "min_O2Sat": patient["O2Sat"].min(),
        "mean_Resp": patient["Resp"].mean(),
        "mean_MAP": patient["MAP"].mean()
    }

    patients_summary.append(summary)

summary_df = pd.DataFrame(patients_summary)
summary_df

In [ ]:
summary_df["septic"].value_counts()

In [ ]:
summary_df.groupby("septic")[[
    "mean_HR",
    "min_O2Sat",
    "mean_Resp",
    "mean_MAP"
]].mean()

## First prediction setup

Using the whole ICU stay to predict a later event would leak future information. For this first test I separate the first 6 observations from the rest.

In [ ]:
patient_id = "p000001"
patient = pd.read_csv(base_url + patient_id + ".psv", sep="|")

first_6h = patient.head(6)
future = patient.iloc[6:]

display(first_6h[["ICULOS", "HR", "O2Sat", "MAP", "Resp", "SepsisLabel"]])
display(future[["ICULOS", "HR", "O2Sat", "MAP", "Resp", "SepsisLabel"]].head())

In [ ]:
early_sepsis = first_6h["SepsisLabel"].max()
future_sepsis = future["SepsisLabel"].max()

print("Positive label in first 6 observations:", early_sepsis)
print("Positive label later in the stay:", future_sepsis)

## Features from the first 6 observations

In [ ]:
mean_hr_6h = first_6h["HR"].mean()
min_o2_6h = first_6h["O2Sat"].min()
mean_resp_6h = first_6h["Resp"].mean()
mean_map_6h = first_6h["MAP"].mean()

print("Mean HR:", mean_hr_6h)
print("Minimum O2Sat:", min_o2_6h)
print("Mean Resp:", mean_resp_6h)
print("Mean MAP:", mean_map_6h)

In [ ]:
hr_min_6h = first_6h["HR"].min()
hr_max_6h = first_6h["HR"].max()
hr_std_6h = first_6h["HR"].std()

print("HR minimum:", hr_min_6h)
print("HR maximum:", hr_max_6h)
print("HR standard deviation:", hr_std_6h)

### HR change with missing values

The first HR can be missing, so I use the first and last available HR inside the window.

In [ ]:
valid_hr = first_6h["HR"].dropna()

if len(valid_hr) >= 2:
    hr_start = valid_hr.iloc[0]
    hr_end = valid_hr.iloc[-1]
    hr_change = hr_end - hr_start
else:
    hr_start = float("nan")
    hr_end = float("nan")
    hr_change = float("nan")

print("First available HR:", hr_start)
print("Last available HR:", hr_end)
print("HR change:", hr_change)

Next step: turn this into a reusable function and define a fixed prediction window.